# 08 — Steering directions and steering strength

**Part A: CPU, minutes. Part B: GPU, hours.**

**Part A** computes one steering direction per model: mean(revision windows) − mean(filler windows) at the centre token, at layer `round(0.65 × (n_layers − 1))` (20 / 18 / 18 / 31 for Llama-8B / Qwen-1.5B / Qwen-7B / Qwen-14B). Writes `hand_labeling_export/steering_directions.json` with the unit direction, the raw (pre-normalisation) norm and the mean activation norm at that layer.

**Part B** sweeps the steering strength α = k × raw direction norm, k ∈ {0, 0.25, …, 6}, on a few events per model, and reports answer-extraction rate (coherence) and the rate of revision-type wording. It does **not** choose k: the values used in the paper (1.5 / 3.0 / 1.5 / 5.0) were chosen by hand from this table and are fixed in `CHOSEN_ALPHA_MULTIPLIER` in `09_necessity_rerun.py`.

Part B reads `hand_labeling_export/steering_pilot_events.json`, the small event set used for calibration. That file is an input to this repository (see README, "Data").

In [ ]:
# ---- Setup: Google Drive and the data folder ---------------------------------
import os
from pathlib import Path
from google.colab import drive
drive.mount("/content/gdrive")

# Folder on Drive that holds traces/, labels/, activations/, rejects/ and
# hand_labeling_export/. Change it here if yours lives elsewhere.
DATA_ROOT = Path(os.environ.get("REVISION_DATA_ROOT",
                                "/content/gdrive/MyDrive/llm_planning_study"))
print("DATA_ROOT =", DATA_ROOT)

## Part A — steering directions

In [ ]:
# =============================================================================
# EXTRACT STEERING DIRECTIONS -- diff-of-means from existing saved activations
#
# direction = mean(revision_windows[center]) - mean(filler_windows[center])
#
# Uses revision_windows (activations/) and filler_windows (activations_filler/)
# -- the SAME classes as the probe's MATCHED contrast (revision-marker vs
# filler-marker, same lexical marker family) -- not the generic
# 'control_windows' field also present in activations/*.npz, which is a
# weaker, non-lexically-matched contrast. Staying consistent with the
# project's established lexical-confound-aware approach.
#
# Layer: does NOT depend on the probe's saved results JSON (path/existence
# unconfirmed, would be a fragile dependency). Instead uses a simple,
# self-contained heuristic grounded in the already-established finding that
# the accuracy plateau begins ~32-38% of network depth across all four
# models: picks a layer at 65% depth (comfortably inside the plateau, not
# near either edge). n_layers is read directly from the activation array
# shape, so this needs no external file at all.
#
# Output: steering_directions.json -- one entry per model with the direction
# (unit-normalized), the chosen layer, and the raw (pre-normalization) norm
# and mean per-position activation norm at that layer (both useful later for
# picking sensible alpha values relative to the activation scale).
# =============================================================================

from pathlib import Path
import json
import numpy as np

GDRIVE_BASE = DATA_ROOT
ACTIV_BASE = GDRIVE_BASE / "activations"
FILLER_BASE = GDRIVE_BASE / "activations_filler"
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"
OUT_PATH = EXPORT_DIR / "steering_directions.json"

MODELS = ["DeepSeek-R1-Distill-Llama-8B", "DeepSeek-R1-Distill-Qwen-1.5B",
          "DeepSeek-R1-Distill-Qwen-7B", "DeepSeek-R1-Distill-Qwen-14B"]
CENTER = 30                # same center-token convention used throughout this project
PLATEAU_DEPTH_FRACTION = 0.65  # heuristic: comfortably inside the ~32-38%-onset plateau


def peek_n_layers(base_dir, model, key):
    """Reads just ONE file's shape to determine n_layers, without loading
    everything. Needed before we know which layer to slice down to."""
    model_dir = base_dir / model
    if not model_dir.exists():
        return None
    for f in sorted(model_dir.glob("*.npz")):
        with np.load(f) as z:
            if key in z:
                return z[key].shape[1]
    return None


def load_sliced_windows(base_dir, model, key, layer, center):
    """Loads each file's array, immediately slices down to [:, layer, center, :],
    and discards the full array before moving to the next file. Peak memory is
    one file's full array at a time (transient), not every file's full array
    held simultaneously -- the previous version's np.concatenate(arrays) after
    the loop kept everything in memory at once, which is what blew up on
    Qwen-14B (48 layers x 5120 hidden dim -- ~51GB combined across both
    revision and filler arrays, held simultaneously, well past typical Colab
    RAM). This version only ever holds the small (n_events_in_file, hidden_dim)
    slices, which are orders of magnitude smaller."""
    slices = []
    model_dir = base_dir / model
    if not model_dir.exists():
        return None
    for f in sorted(model_dir.glob("*.npz")):
        with np.load(f) as z:
            if key not in z:
                continue
            arr = z[key]                       # full array for this file, transient
            sliced = arr[:, layer, center, :].astype(np.float32).copy()
            slices.append(sliced)
            del arr                            # explicit, though scope-exit would do this anyway
    if not slices:
        return None
    return np.concatenate(slices, axis=0)      # concatenating small slices is cheap


def extract_direction_for_model(model):
    n_layers = peek_n_layers(ACTIV_BASE, model, "revision_windows")
    if n_layers is None:
        print(f"  {model}: no revision_windows files found -- skipping")
        return None

    layer = int(round(PLATEAU_DEPTH_FRACTION * (n_layers - 1)))
    layer = max(0, min(layer, n_layers - 1))

    rev_vecs = load_sliced_windows(ACTIV_BASE, model, "revision_windows", layer, CENTER)
    fil_vecs = load_sliced_windows(FILLER_BASE, model, "filler_windows", layer, CENTER)

    if rev_vecs is None or fil_vecs is None:
        print(f"  {model}: missing revision_windows or filler_windows -- skipping")
        return None

    rev_mean = rev_vecs.mean(axis=0)
    fil_mean = fil_vecs.mean(axis=0)
    raw_direction = rev_mean - fil_mean
    raw_norm = float(np.linalg.norm(raw_direction))

    if raw_norm == 0:
        print(f"  {model}: degenerate zero-norm direction at layer {layer} -- skipping")
        return None

    unit_direction = raw_direction / raw_norm

    # mean activation norm at this layer/position -- reference scale for alpha calibration
    all_vecs = np.concatenate([rev_vecs, fil_vecs], axis=0)
    mean_activation_norm = float(np.linalg.norm(all_vecs, axis=1).mean())

    return {
        "model": model,
        "n_layers": n_layers,
        "layer": layer,
        "layer_fraction": layer / (n_layers - 1),
        "n_revision_events": int(rev_vecs.shape[0]),
        "n_filler_events": int(fil_vecs.shape[0]),
        "raw_direction_norm": raw_norm,
        "mean_activation_norm_at_layer": mean_activation_norm,
        "unit_direction": unit_direction.tolist(),
    }


def main():
    results = {}
    for model in MODELS:
        print(f"\n{model}")
        r = extract_direction_for_model(model)
        if r is not None:
            results[model] = r
            print(f"  layer={r['layer']}/{r['n_layers']} ({r['layer_fraction']:.0%} depth)  "
                  f"n_revision={r['n_revision_events']}  n_filler={r['n_filler_events']}")
            print(f"  raw direction norm={r['raw_direction_norm']:.3f}  "
                  f"mean activation norm at this layer={r['mean_activation_norm_at_layer']:.3f}")
            print(f"  (ratio: direction is {r['raw_direction_norm']/r['mean_activation_norm_at_layer']:.1%} "
                  f"of typical activation magnitude at this layer -- a sanity check, not itself alpha)")

    with open(OUT_PATH, "w") as f:
        json.dump(results, f)
    print(f"\nWrote {len(results)} model directions to {OUT_PATH}")


if __name__ == "__main__":
    main()

## Part B — calibrate the steering strength

In [ ]:
# =============================================================================
# CALIBRATE STEERING ALPHA
#
# Tests a few candidate steering strengths (including alpha=0, no steering, as
# a baseline) on a small number of events per model. Reports, per (model,
# alpha): answer-extraction success rate (coherence proxy -- if this craters,
# generation broke, not a finding about necessity) and the NATURAL rate of
# backtracking-adjacent language (using the same LEAKAGE_FLAG_WORDS list from
# the lexical-suppression pipeline, observed here, not enforced).
#
# This does NOT auto-select alpha. Per the design doc: read the printed table,
# pick the smallest alpha that meaningfully reduces flagged-language rate
# while keeping the extraction-success rate close to the alpha=0 baseline,
# and record that choice (with this table) before running the real pilot.
#
# Deliberately cheap: short generations (fewer tokens than the real pilot),
# few events, few rollouts. This is exploratory, not a result.
#
# ALPHA SCALE: relative to raw_direction_norm (the natural revision-vs-filler
# signal magnitude), NOT mean_activation_norm_at_layer (ambient scale). These
# differ by ~8-10x for these models -- scaling against the ambient norm meant
# even the gentlest originally-swept value was already several multiples of
# the natural signal. Fixed after review; the sweep is now anchored correctly
# and widened (8 points instead of 4) to actually map the coherence/effect
# tradeoff rather than jumping from "too weak to matter" to "far too strong".
# =============================================================================

from pathlib import Path
import json, re, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GDRIVE_BASE = DATA_ROOT
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"
STEERING_PILOT_EVENTS_PATH = EXPORT_DIR / "steering_pilot_events.json"
STEERING_DIRECTIONS_PATH = EXPORT_DIR / "steering_directions.json"
CALIBRATION_OUT_PATH = EXPORT_DIR / "steering_alpha_calibration.json"

N_EVENTS_PER_MODEL = 2       # keep this small -- calibration is exploratory
N_ROLLOUTS_PER_CANDIDATE = 2
CALIBRATION_MAX_NEW_TOKENS = 2048   # matches ROLLOUT_MAX_NEW_TOKENS in generate_steering_rollouts.py.
# Previously 512 -- but extraction succeeded only 0-25% of the time even at
# alpha=0 (no steering at all), meaning the short budget alone was starving
# completions regardless of steering. That made every extraction-rate number
# uninterpretable: no way to tell "steering broke coherence" from "ran out of
# tokens". Matching the real pilot's budget removes that confound, at the
# cost of calibration no longer being meaningfully cheaper than the real run.
ALPHA_MULTIPLIERS = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0]
# Extended upward from the original [.. 3.0] max after that sweep showed
# 'wait' stuck near 100% across the whole range while coherence (extraction
# rate) hadn't degraded -- real headroom to test whether 'wait' ever moves at
# higher strength, or whether it's stuck regardless (evidence it may be a
# generic discourse habit rather than something a single direction controls).
# x raw_direction_norm (the NATURAL magnitude of the revision-vs-filler difference),
# NOT mean_activation_norm_at_layer. raw_direction_norm is only ~10-12% of the ambient
# activation scale for these models -- scaling alpha against the ambient norm meant
# even the smallest previously-swept value (0.5x mean_norm) was already ~4-5x the
# natural signal, not a gentle probe. multiplier=1.0 here means "push by exactly the
# natural class-mean difference" -- a meaningful, interpretable reference point.
# 0.0 = no-steering baseline.

LEAKAGE_FLAG_WORDS = [
    "verify", "double-check", "double check", "seems off", "seems wrong",
    "let me check", "re-examine", "reexamine", "start over", "scratch that",
    "different approach", "different method", "alternative approach", "instead, let's",
    "i realize", "i notice", "hold on", "not quite right", "doesn't seem",
    "let's redo", "recompute", "re-derive", "wait", "actually", "alternatively",
    "hmm", "let me reconsider", "i made a mistake", "let me try again",
]


def extract_answer(text: str) -> str:
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        return boxed[-1].strip()
    m = re.search(r'(?:answer is|therefore)[:\s]*([0-9A-Za-z\-\.\/]+)', text, re.IGNORECASE)
    return m.group(1).strip() if m else "UNKNOWN"


def build_prompt(problem: dict, tokenizer, model_name: str) -> str:
    is_reasoning = any(tag in model_name for tag in ["R1", "QwQ"])
    if problem["difficulty"] == "hard":
        instruction = ("Solve the following competition mathematics problem carefully. "
                        "Show your reasoning step by step and verify your answer.")
    else:
        instruction = ("Solve the following mathematics competition problem. "
                        "Show your working step by step.")
    if is_reasoning:
        messages = [{"role": "user", "content": f"{instruction}\n\nProblem: {problem['problem']}"}]
    else:
        messages = [
            {"role": "system", "content": "You are an expert mathematician. Solve problems with "
                                          "clear step-by-step reasoning."},
            {"role": "user", "content": f"{instruction}\n\nProblem: {problem['problem']}"}
        ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def flag_leakage_rate(text):
    text_lower = text.lower()
    return any(w.lower() in text_lower for w in LEAKAGE_FLAG_WORDS)


def matched_leakage_words(text):
    text_lower = text.lower()
    return [w for w in LEAKAGE_FLAG_WORDS if w.lower() in text_lower]


def get_decoder_layers(model):
    """Locates the decoder layer list for standard HF Llama/Qwen2-style models.
    Both architectures used in this project follow model.model.layers -- fails
    loudly with a clear message if that assumption doesn't hold, rather than
    silently hooking the wrong thing."""
    assert hasattr(model, "model") and hasattr(model.model, "layers"), (
        "Expected model.model.layers (standard HF Llama/Qwen2 layout) -- this model's "
        "architecture doesn't match. Inspect model structure manually before proceeding."
    )
    return model.model.layers


def make_steering_hook(direction_tensor, alpha):
    def hook(module, inputs, output):
        if isinstance(output, tuple):
            hidden_states = output[0] - alpha * direction_tensor
            return (hidden_states,) + output[1:]
        return output - alpha * direction_tensor
    return hook


@torch.no_grad()
def generate_with_steering(model, tokenizer, prefix_text, layer_idx, direction_tensor, alpha,
                            max_new_tokens, seed):
    torch.manual_seed(seed)
    inputs = tokenizer(prefix_text, return_tensors="pt", truncation=False).to(model.device)

    handle = None
    if alpha != 0.0:
        layers = get_decoder_layers(model)
        handle = layers[layer_idx].register_forward_hook(make_steering_hook(direction_tensor, alpha))

    try:
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=0.6, top_p=0.95,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    finally:
        if handle is not None:
            handle.remove()  # always remove, even if generation raises

    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=False)


def main():
    with open(STEERING_PILOT_EVENTS_PATH) as f:
        pilot_events = json.load(f)
    with open(STEERING_DIRECTIONS_PATH) as f:
        directions = json.load(f)

    by_model = {}
    for ev in pilot_events:
        by_model.setdefault(ev["model"], []).append(ev)

    calibration_results = []

    for model_name_short, events in by_model.items():
        if model_name_short not in directions:
            print(f"\n{model_name_short}: no steering direction available -- skipping calibration")
            continue
        dir_info = directions[model_name_short]
        layer_idx = dir_info["layer"]
        natural_norm = dir_info["raw_direction_norm"]  # reference scale for alpha -- see ALPHA_MULTIPLIERS comment
        mean_norm = dir_info["mean_activation_norm_at_layer"]  # reported for context only, not used for alpha

        model_name_full = f"deepseek-ai/{model_name_short}"
        print(f"\n{'='*60}\n{model_name_full}  (layer {layer_idx}, natural signal norm {natural_norm:.2f}, "
              f"ambient activation norm {mean_norm:.2f})\n{'='*60}")

        tokenizer = AutoTokenizer.from_pretrained(model_name_full, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_name_full, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
        model.eval()

        direction_tensor = torch.tensor(dir_info["unit_direction"], dtype=torch.float16, device=model.device)

        calib_events = events[:N_EVENTS_PER_MODEL]
        word_counts_by_alpha = {}  # alpha_multiplier -> {word: count}
        for mult in ALPHA_MULTIPLIERS:
            alpha = mult * natural_norm
            n_extracted, n_flagged, n_total = 0, 0, 0
            word_counts = {}
            print(f"\n  alpha = {mult} x natural_signal_norm = {alpha:.3f}")
            for ev in calib_events:
                problem_dict = {"problem": ev["problem"], "difficulty": ev["difficulty"]}
                prompt = build_prompt(problem_dict, tokenizer, model_name_full)
                prefix_text = prompt + ev["prefix_response_text"]
                for r in range(N_ROLLOUTS_PER_CANDIDATE):
                    text = generate_with_steering(
                        model, tokenizer, prefix_text, layer_idx, direction_tensor, alpha,
                        CALIBRATION_MAX_NEW_TOKENS, seed=2000 + r)
                    ans = extract_answer(text)
                    n_total += 1
                    if ans != "UNKNOWN":
                        n_extracted += 1
                    matches = matched_leakage_words(text)
                    if matches:
                        n_flagged += 1
                        for w in matches:
                            word_counts[w] = word_counts.get(w, 0) + 1
            extraction_rate = n_extracted / n_total
            flagged_rate = n_flagged / n_total
            word_counts_by_alpha[mult] = word_counts
            top_words = sorted(word_counts.items(), key=lambda kv: -kv[1])[:5]
            print(f"    extraction success rate: {extraction_rate:.0%}   "
                  f"backtracking-language rate: {flagged_rate:.0%}   (n={n_total} rollouts)")
            print(f"    top matched words: {top_words}")
            calibration_results.append({
                "model": model_name_short, "alpha_multiplier": mult, "alpha": alpha,
                "extraction_success_rate": extraction_rate, "flagged_language_rate": flagged_rate,
                "n_rollouts": n_total, "word_counts": word_counts,
            })

        # Per-model summary: for each flag word, how did its rate change from
        # alpha=0 to the strongest alpha tested? This is the actual answer to
        # "is the 100% ceiling generic words or real backtracking language".
        print(f"\n  --- {model_name_short}: per-word rate at alpha=0 vs. strongest alpha tested ---")
        baseline_counts = word_counts_by_alpha.get(0.0, {})
        strongest_mult = max(ALPHA_MULTIPLIERS)
        strongest_counts = word_counts_by_alpha.get(strongest_mult, {})
        n_baseline_rollouts = N_EVENTS_PER_MODEL * N_ROLLOUTS_PER_CANDIDATE
        all_words_seen = set(baseline_counts) | set(strongest_counts)
        for w in sorted(all_words_seen, key=lambda w: -baseline_counts.get(w, 0)):
            b = baseline_counts.get(w, 0) / n_baseline_rollouts
            s = strongest_counts.get(w, 0) / n_baseline_rollouts
            print(f"    {w!r:<28} alpha=0: {b:.0%}   alpha={strongest_mult}: {s:.0%}")


        del model
        torch.cuda.empty_cache()

    with open(CALIBRATION_OUT_PATH, "w") as f:
        json.dump(calibration_results, f, indent=2)

    print(f"\n{'#'*70}\n# CALIBRATION SUMMARY -- read this table, then pick alpha manually\n{'#'*70}")
    print("alpha_mult is relative to the NATURAL signal norm (raw_direction_norm), not ambient")
    print("activation magnitude -- 1.0 means 'push by exactly the natural revision-vs-filler")
    print("class-mean difference'.\n")
    print(f"{'Model':<28} {'alpha_mult':>10} {'extraction_rate':>16} {'flagged_rate':>13}")
    for r in calibration_results:
        print(f"{r['model']:<28} {r['alpha_multiplier']:>10} "
              f"{r['extraction_success_rate']:>15.0%} {r['flagged_language_rate']:>12.0%}")
    print(f"\nWrote {CALIBRATION_OUT_PATH}")
    print("\nPick the smallest alpha_multiplier where flagged_rate drops meaningfully below the")
    print("alpha=0 baseline WITHOUT extraction_rate collapsing -- record your choice before Step 4.")


if __name__ == "__main__":
    main()